# Compare — Aligned vs Naive feature 비교

`01-registration.ipynb` (aligned) 와 `02-naive-baseline.ipynb` (naive) 에서 저장한 `features.csv` / `features_naive.csv` 를 불러와서 나란히 시각화.

**전제**: 두 노트북 다 먼저 실행돼서 CSV 두 개가 있어야 함.

**비교 대상**:
- Timeline plot (metric × patient × eye) — 두 파이프라인 겹쳐 그리기
- CV (coefficient of variation) 표 + bar chart
- Metric 별 산점도 (naive vs aligned per-visit)


## 0. Imports + 경로


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SQUARE_SIZE = 512
src_name = "20230118_HCS_Fundus_image"

aligned_csv = Path(f"g:/RAG/project_1/output/aligned_{SQUARE_SIZE}") / src_name / "features.csv"
naive_csv   = Path(f"g:/RAG/project_1/output/naive_{SQUARE_SIZE}")   / src_name / "features_naive.csv"
compare_out = Path(f"g:/RAG/project_1/output/compare_{SQUARE_SIZE}") / src_name
compare_out.mkdir(parents=True, exist_ok=True)

assert aligned_csv.exists(), f"먼저 01-registration.ipynb 실행 → {aligned_csv}"
assert naive_csv.exists(),   f"먼저 02-naive-baseline.ipynb 실행 → {naive_csv}"
print("aligned:", aligned_csv)
print("naive:  ", naive_csv)


## 1. Features 로드


In [ ]:
features_aligned = pd.read_csv(aligned_csv, parse_dates=["date"])
features_naive   = pd.read_csv(naive_csv,   parse_dates=["date"])

s1, s2 = set(features_naive["id"]), set(features_aligned["id"])
common_ids = s1 & s2
print(f"aligned rows: {len(features_aligned)}, naive rows: {len(features_naive)}")
print(f"공통 id: {len(common_ids)}, naive-only: {len(s1-s2)}, aligned-only: {len(s2-s1)}")

patient_ids = sorted(features_aligned["patient"].unique())
print("patients:", patient_ids)


## 2. Side-by-side timeline

실선 = aligned, 점선 = naive. L/R 눈 각각.


In [ ]:
metrics = ["vessel_density", "artery_density", "vein_density", "AV_ratio_area", "disc_frac"]
n_pat = len(patient_ids)

fig, axes = plt.subplots(n_pat, len(metrics), figsize=(4*len(metrics), 3.4*n_pat), squeeze=False)
for row, pid in enumerate(patient_ids):
    for col, metric in enumerate(metrics):
        ax = axes[row, col]
        for eye, style_a, style_n in [("L", "-o", "--o"), ("R", "-s", "--s")]:
            a = features_aligned[(features_aligned["patient"]==pid) & (features_aligned["eye"]==eye)]
            n = features_naive  [(features_naive  ["patient"]==pid) & (features_naive  ["eye"]==eye)]
            if len(a):
                ax.plot(a["date"], a[metric], style_a,
                        label=f"{eye} aligned", markersize=4, linewidth=1.4, alpha=0.9)
            if len(n):
                ax.plot(n["date"], n[metric], style_n,
                        label=f"{eye} naive",   markersize=3, linewidth=0.9, alpha=0.7, color="gray")
        ax.set_title(f"{pid} — {metric}")
        ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
        ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout(); plt.show()


## 3. CV 정량 비교

`CV = std / mean × 100` (%). 낮을수록 fluctuation 적음.


In [ ]:
def cv_by_group(df, metric):
    g = df.groupby(["patient", "eye"])[metric]
    return 100 * g.std() / g.mean()

rows = []
for metric in metrics:
    cv_a = cv_by_group(features_aligned, metric)
    cv_n = cv_by_group(features_naive,   metric)
    for (pid, eye), cv_val in cv_a.items():
        rows.append({
            "patient": pid, "eye": eye, "metric": metric,
            "CV_aligned": cv_val,
            "CV_naive":   cv_n.get((pid, eye), np.nan),
        })
cv_df = pd.DataFrame(rows)
cv_df["delta_CV"]     = cv_df["CV_naive"] - cv_df["CV_aligned"]
cv_df["reduction_%"]  = 100 * cv_df["delta_CV"] / cv_df["CV_naive"]
cv_df.to_csv(compare_out / "cv_comparison.csv", index=False)
cv_df.round(2)


### CV bar chart — metric 별 평균


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_pivot = cv_df.pivot_table(index="metric", values=["CV_naive", "CV_aligned"], aggfunc="mean")
cv_pivot = cv_pivot.reindex(metrics)   # metric 순서 고정
# naive 앞, aligned 뒤 (column 순서로 제어)
cv_pivot = cv_pivot[["CV_naive", "CV_aligned"]]
cv_pivot.plot(kind="bar", ax=ax, color=["tab:gray", "tab:blue"])
ax.set_ylabel("CV (%) — 평균 across (patient, eye)")
ax.set_title("Fluctuation 비교: Naive vs Aligned")
ax.grid(alpha=0.3, axis="y")
ax.tick_params(axis="x", rotation=15)
for i, m in enumerate(cv_pivot.index):
    n, a = cv_pivot.loc[m, "CV_naive"], cv_pivot.loc[m, "CV_aligned"]
    reduction = 100 * (n - a) / n if n else 0
    ax.text(i, max(a, n) + 0.3,
            f"{'↓' if reduction>0 else '↑'}{abs(reduction):.0f}%",
            ha="center", fontsize=9,
            color="green" if reduction>0 else "red")
plt.tight_layout(); plt.show()


## 4. Per-visit 산점도 — aligned vs naive

대각선 근처에 있으면 두 방법 결과가 비슷 (metric 이 alignment 무관).  
대각선에서 멀수록 alignment 가 값을 크게 바꿈 (mask 위치 의존적).


In [ ]:
merged = features_naive.merge(features_aligned, on=["id", "patient", "eye", "date", "shot"],
                              suffixes=("_naive", "_aligned"))

fig, axes = plt.subplots(1, len(metrics), figsize=(4*len(metrics), 4))
for ax, metric in zip(axes, metrics):
    x = merged[f"{metric}_naive"]
    y = merged[f"{metric}_aligned"]
    ax.scatter(x, y, s=10, alpha=0.6, c=merged["patient"].astype("category").cat.codes, cmap="tab10")
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.4, linewidth=1, label="y=x")
    ax.set_xlabel(f"{metric} (naive)"); ax.set_ylabel(f"{metric} (aligned)")
    ax.set_title(metric); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 5. 요약

- **`cv_df`** — (patient, eye, metric) 별 CV 비교, `compare_out/cv_comparison.csv` 로 저장
- **해석 가이드**:
  - `Aligned CV < Naive CV` → alignment 가 fluctuation 을 실제로 줄임
  - `Aligned CV ≈ Naive CV` → 이 metric 은 alignment 무관 (예: total density 처럼 위치 안 타는 지표)
  - `Aligned CV > Naive CV` → alignment 가 오히려 노이즈 도입 (드묾, warp/undo 오차 커지는 경우)
